# Agent Memory Systems

One characteristic of LLM & retrieval systems is that the entire process is **stateless**, e.g. it cannot learn from interactions. Retrieval systems generally involve queries to an external data source, then adding the response to the current model context. In particular, the language model begins each new interaction with a fresh state, which can be limiting for personalization or building rich context histories. Hence, we want to create a **memory system** for our agents.

That said, the core LLM remains fundamentally stateless. To address this, we structure the context so that relevant memories are dynamically loaded into the chat history during interactions with the LLM. This is similar to reminding a memory-less LLM of certain facts about yourself at the start of each new conversation except it is designed in a systematic way that makes it effective off-the-shelf. Moreover, since memory now affects future generation, we need to design a way to **systematically forget** or **update** memories. Otherwise, the context will be overloaded with outdated facts and reflections[^reflection]. 

[^reflection]: It is recommended to read the [@genagents] paper along with this notebook.

In [25]:
from notebooks.utils import load_dotenv
from notebooks.agents.utils import get_client
from notebooks.agents.chat import ChatHistory, ChatCompletions, Deployment

load_dotenv()
MODEL = "openai/gpt-oss-20b"
client = get_client("groq")
deployment = Deployment(client, MODEL)

In [21]:
import json
from typing import Any, Optional

class MemoryStore:
    def __init__(self, path: str = "memory_store.json"):
        self.path = path
        self._load()

    def _load(self):
        try:
            with open(self.path, "r", encoding="utf-8") as f:
                self.store = json.load(f)
        except FileNotFoundError:
            self.store = []  # list of dicts: {id, category, text, metadata}

    def _save(self):
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(self.store, f, ensure_ascii=False, indent=2)

    def add(self, item: Dict[str, Any]) -> Dict[str, Any]:
        # Ensure minimal schema
        item = item.copy()
        if "id" not in item:
            item["id"] = str(len(self.store) + 1)
        self.store.append(item)
        self._save()
        return item

    def update(self, item_id: str, updates: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        for it in self.store:
            if it.get("id") == item_id:
                it.update(updates)
                self._save()
                return it
        return None

    def delete(self, item_id: str) -> bool:
        for i, it in enumerate(self.store):
            if it.get("id") == item_id:
                del self.store[i]
                self._save()
                return True
        return False

    def search(self, query: str) -> List[Dict[str, Any]]:
        # Simple substring search over text and category
        q = query.lower()
        return [it for it in self.store if q in (it.get("text","")+it.get("category","")).lower()]

    def all(self) -> List[Dict[str, Any]]:
        return list(self.store)

# initialize
memory_store = MemoryStore()


In [23]:
SENTINEL_SYSTEM = """
You are a classifier whose only job is to decide whether a short user message contains *new information worth storing* in a family's profile knowledge base used for long-term personalization.

Return exactly one token: TRUE or FALSE (capitalized). No extra text, no punctuation, no explanation.

Only consider storing facts that are likely to be useful later (preferences, allergies, household composition, location, routines, stable facts). Do NOT store ephemeral chit-chat or greetings.
"""

def memory_sentinel(user_message: str) -> str:
    messages = [
        {"role": "system", "content": SENTINEL_SYSTEM},
        {"role": "user", "content": user_message},
    ]
    out = call_chat(messages, temperature=0.0)
    # Normalize
    out = out.strip().upper()
    if out not in {"TRUE", "FALSE"}:
        # If model deviates, apply a simple heuristic fallback using keywords
        keywords = ["allerg", "like", "dislike", "hate", "live", "from", "children", "husband", "wife", "pregnant"]
        if any(k in user_message.lower() for k in keywords):
            return "TRUE"
        return "FALSE"
    return out

In [24]:
MANAGER_SYSTEM = """
You are an assistant that extracts structured memory items from a short user message.\
Respond with a valid JSON object ONLY (no surrounding text). The schema:
{
  "action": "add"|"update"|"delete"|"none",
  "item": {"id": optional string, "category": string, "text": string, "metadata": { ... }}
}

Rules:
- action `add` means create a new memory item.
- action `update` means modify an existing item; include `id` in item.
- action `delete` means delete an existing item; include `id` and can keep text terse.
- action `none` means don't change the store.

Be conservative about creating new items. If the message is vague or ambiguous, choose `none`.
"""

def memory_manager(user_message: str) -> Dict[str, Any]:
    messages = [
        {"role": "system", "content": MANAGER_SYSTEM},
        {"role": "user", "content": user_message},
    ]
    out = call_chat(messages, temperature=0.0)
    # Model should return JSON. Try to parse.
    try:
        parsed = json.loads(out)
    except Exception:
        # Fallback: attempt to extract by prompting the model to *only* return JSON again.
        messages.append({"role": "assistant", "content": out})
        messages.append({"role": "system", "content": "Previous response was not valid JSON. Return only a JSON object that follows the described schema."})
        out2 = call_chat(messages, temperature=0.0)
        parsed = json.loads(out2)
    return parsed


In [25]:
def execute_action(action_obj: Dict[str, Any], store: MemoryStore) -> Dict[str, Any]:
    action = action_obj.get("action")
    item = action_obj.get("item") or {}

    if action == "add":
        return {"status": "added", "item": store.add(item)}
    elif action == "update":
        item_id = item.get("id")
        if not item_id:
            return {"status": "error", "reason": "no id provided for update"}
        updated = store.update(item_id, item)
        if updated:
            return {"status": "updated", "item": updated}
        return {"status": "error", "reason": "id not found"}
    elif action == "delete":
        item_id = item.get("id")
        if not item_id:
            return {"status": "error", "reason": "no id provided for delete"}
        ok = store.delete(item_id)
        return {"status": "deleted" if ok else "error", "id": item_id}
    elif action == "none":
        return {"status": "none"}
    else:
        return {"status": "error", "reason": "unknown action"}


In [26]:
conversation_history: List[Dict[str, str]] = [
    {"role": "system", "content": "You are part of a memory pipeline. Keep replies short."}
]

def process_user_message(user_message: str):
    # append to conversation history
    conversation_history.append({"role": "user", "content": user_message})

    # 1. Sentinel
    should_store = memory_sentinel(user_message)

    if should_store == "FALSE":
        assistant_reply = call_chat(conversation_history + [{"role": "system", "content": "No memory will be stored."}], temperature=0.2)
        conversation_history.append({"role": "assistant", "content": assistant_reply})
        return {"stored": False, "assistant_reply": assistant_reply}

    # 2. Manager
    manager_out = memory_manager(user_message)

    # 3. Executor
    exec_res = execute_action(manager_out, memory_store)

    # 4. Respond to user
    assistant_reply = f"Memory action: {exec_res.get('status')}"
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    return {"stored": True, "manager_out": manager_out, "exec_res": exec_res, "assistant_reply": assistant_reply}


In [27]:
examples = [
    "My wife is allergic to peanuts.",
    "We love eating pasta and prefer vegetarian options on Mondays.",
    "Hi, how are you?",
    "Update: her allergy is actually tree nuts, not peanuts."
]

for ex in examples:
    print("---\nUser:", ex)
    out = process_user_message(ex)
    print(out)

print('\nCurrent memory store:')
print(memory_store.all())


---
User: My wife is allergic to peanuts.
{'stored': True, 'manager_out': {'action': 'add', 'item': {'category': 'allergy', 'text': 'My wife is allergic to peanuts.', 'metadata': {'relation': 'wife'}}}, 'exec_res': {'status': 'added', 'item': {'category': 'allergy', 'text': 'My wife is allergic to peanuts.', 'metadata': {'relation': 'wife'}, 'id': '1'}}, 'assistant_reply': 'Memory action: added'}
---
User: We love eating pasta and prefer vegetarian options on Mondays.
{'stored': True, 'manager_out': {'action': 'add', 'item': {'category': 'food_preference', 'text': 'We love eating pasta and prefer vegetarian options on Mondays.', 'metadata': {'preference': 'pasta', 'vegetarian_on': ['Monday']}}}, 'exec_res': {'status': 'added', 'item': {'category': 'food_preference', 'text': 'We love eating pasta and prefer vegetarian options on Mondays.', 'metadata': {'preference': 'pasta', 'vegetarian_on': ['Monday']}, 'id': '2'}}, 'assistant_reply': 'Memory action: added'}
---
User: Hi, how are you?


In [127]:
%reload_ext autoreload
%autoreload 2
import importlib
import notebooks
importlib.reload(notebooks.agents.chat)

<module 'notebooks.agents.chat' from '/Users/particle1331/code/ai-notebooks/src/notebooks/agents/chat.py'>

In [128]:
from notebooks.agents.chat import ChatHistory, ChatCompletions
from notebooks.agents.utils import get_client, Deployment
from notebooks.utils import load_dotenv, notna

In [129]:
load_dotenv(verbose=False)
client = get_client("groq")
MODEL = "openai/gpt-oss-20b"
completions = ChatCompletions(Deployment(client, MODEL))

## Working memory

[@liu2023lostmiddlelanguagemodels]

![[@liu2023lostmiddlelanguagemodels]](./img/lost-in-the-middle.jpg){#fig-liu2023}

active working memory:

In [5]:
chat = ChatHistory("You are a helpful assistant. Answer succintly in one sentence.")
while True:
    user_message = input()
    print("HUMAN:\t", user_message)
    if user_message == "exit":
        break

    chat.update(role="user", prompt=user_message)
    response = completions.create(chat, temperature=1.0)
    chat.update(role="assistant", prompt=response)

    print("AI:\t\t", response)

HUMAN:	 What is my name?
AI:		 I’m sorry, I don’t know your name.
HUMAN:	 My name is Iilie.
AI:		 Nice to meet you, Iilie!
HUMAN:	 Cool! Do you now know what my name is?
AI:		 Yes, your name is Iilie.
HUMAN:	 exit


## Episodic memory

episodic memory = historical collection of prior experiences or episodes

non-explicitly stated take-aways including literal recollection.

dynamic few shot prompting. examples, instructions as messages come in.

run generation step where raw conversation are stored and specific reflections.

experiential foundation. proven interaction patterns.

### Reflection chain

reflection:

- context tags
- conversation summary
- what worked
- what to avoid

basically a review of the conversation.

In [7]:
from pydantic import BaseModel, Field
from typing import Annotated

class Reflection(BaseModel):
    context_tags:          Annotated[list[str], Field(description="Keywords that would help identify similar future conversations. Use field-specific terms like 'deep_learning', 'methodology_question', 'results_interpretation'", min_length=1, max_length=4)]
    conversation_summary:  Annotated[str,       Field(description="One sentence describing what the conversation accomplished.")]
    what_worked:           Annotated[str,       Field(description="Most effective approach or strategy used in this conversation.")]
    what_to_avoid:         Annotated[str,       Field(description="Most important pitfall or ineffective approach to avoid.")]


reflection_prompt_template = """
You are analyzing conversations about research papers to create memories that will help guide future interactions. 
Your task is to extract key elements that would be most helpful when encountering similar academic discussions in the future.

Review the conversation and create a memory reflection following these rules:

1. For any field where you don't have enough information or the field isn't relevant, use "N/A"
2. Be extremely concise - each string should be one clear, actionable sentence
3. Focus only on information that would be useful for handling similar future conversations
4. Context_tags should be specific enough to match similar situations but general enough to be reusable

Examples:
- Good context_tags: ["transformer_architecture", "attention_mechanism", "methodology_comparison"]
- Bad  context_tags: ["machine_learning", "paper_discussion", "questions"]

- Good conversation_summary: "Explained how the attention mechanism in the BERT paper differs from traditional transformer architectures"
- Bad  conversation_summary: "Discussed a machine learning paper"

- Good what_worked: "Using analogies from matrix multiplication to explain attention score calculations"
- Bad  what_worked: "Explained the technical concepts well"

- Good what_to_avoid: "Diving into mathematical formulas before establishing user's familiarity with linear algebra fundamentals"
- Bad  what_to_avoid: "Used complicated language"

Additional examples for different research scenarios:

Context tags examples:
- ["experimental_design", "control_groups", "methodology_critique"]
- ["statistical_significance", "p_value_interpretation", "sample_size"]
- ["research_limitations", "future_work", "methodology_gaps"]

Conversation summary examples:
- "Clarified why the paper's cross-validation approach was more robust than traditional hold-out methods"
- "Helped identify potential confounding variables in the study's experimental design"

What worked examples:
- "Breaking down complex statistical concepts using visual analogies and real-world examples"
- "Connecting the paper's methodology to similar approaches in related seminal papers"

What to avoid examples:
- "Assuming familiarity with domain-specific jargon without first checking understanding"
- "Over-focusing on mathematical proofs when the user needed intuitive understanding"

Do not include any text outside the JSON object in your response.

Here is the prior conversation:

{conversation}
"""


In [26]:
reflection_chat = ChatHistory(reflection_prompt_template.format(conversation=chat))
completions.create(reflection_chat, response_format=Reflection)

{'context_tags': ['personal_information',
  'identity_verification',
  'confirmation'],
 'conversation_summary': "Assistant initially did not know the user's name, user provided it, and the assistant confirmed it back.",
 'what_worked': 'Explicitly asked the user for missing information and then repeated it back to confirm understanding.',
 'what_to_avoid': "Assuming the user's name without explicit confirmation."}

Initializing vector database:

In [44]:
import json
import chromadb
from chromadb.utils import embedding_functions

from notebooks.utils import ROOT_PATH

client = chromadb.PersistentClient(path=ROOT_PATH / "topics" / "agents" / ".chroma_db")

ollama_embfunc = embedding_functions.OllamaEmbeddingFunction(
    url="http://localhost:11434",  # Ollama endpoint
    model_name="nomic-embed-text",
)

collection = client.get_or_create_collection(
    name="episodic_memory",
    embedding_function=ollama_embfunc,
    metadata={"description": "Collection containing historical chat interactions and takeaways."}
)

Pushing in documents:

In [ ]:
# Add some data
collection.upsert(
    documents=[
        "Chat about debugging FastAPI and Celeres.",
        "Conversation about neural network saddle points yip."
    ],
    metadatas=[
        {
            "context_tags": json.dumps(["debugging", "fastapi"]),
            "what_worked": "Used logging",
            "what_to_avoid": "Skipping health checks"
        },
        {
            "context_tags": json.dumps(["theory", "deep_learning"]),
            "what_worked": "Visualization",
            "what_to_avoid": "Ignoring plateaus"
        }
    ],
    ids=["1", "2"]
)

# Query
results = collection.query(
    query_texts=["python backend"],
    n_results=1
)

print(results)

{'ids': [['1']], 'embeddings': None, 'documents': [['Chat about debugging FastAPI and Celeres.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'context_tags': '["debugging", "fastapi"]', 'what_worked': 'Used logging', 'what_to_avoid': 'Skipping health checks'}]], 'distances': [[0.5617914795875549]]}


![The LLM expresses the intent to write to the memory store via the structured output. Then, it is up to the main program to perform the actual writing. This allows hooks like [guardrails](https://cookbook.openai.com/examples/how_to_use_guardrails) to be applied before executing the function. Note that the retrieval happens prior to LLM processing. It would be nice to have the LLM read the entire filestore but this becomes more expensive as the memory store grows. In practice, information retrieval techniques such as TF-IDF and embedding similarity can be used.
](./img/retrieval-system.png){#fig-retrieval-system}

**Memory store.** Memories are saved as dictionaries `{"tag": <tag>, "fact": <fact>}`. The following objects work around this definition. A retrieval function is defined as a method of the memory store class which gets relevant memory items based on *keyword search*. Hence, we have the following function for extracting keywords:

In [ ]:
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))

def tokenize(text):
    """Tokenize into words then remove stopwords."""
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    return filtered_tokens

text = "Punk is a pre-trained unsupervised machine learning model for tokenization. It's one of the most crucial and widely used components in the NLTK library."
print("Original:", text)
print("Filtered:", tokenize(text))

Original: Punk is a pre-trained unsupervised machine learning model for tokenization. It's one of the most crucial and widely used components in the NLTK library.
Filtered: ['punk', 'pre-trained', 'unsupervised', 'machine', 'learning', 'model', 'tokenization', "'s", 'one', 'crucial', 'widely', 'used', 'components', 'nltk', 'library']


In [ ]:
import json
from typing import List
from openai import OpenAI
from pydantic import BaseModel


class MemoryItem(BaseModel):
    tag: str
    fact: str
    reason: str

class MemoryResponse(BaseModel):
    items: List[MemoryItem]


class MemoryStore:
    def __init__(self, path="memory.json"):
        """Load memory from JSON file in local path."""
        self.path = path
        self.data = []
        self.tags = set()
        self.load()

    def load(self):
        try:
            self.data = json.load(open(self.path))
        except FileNotFoundError:
            self.reset()

    def save(self):
        with open(self.path, "w") as f:
            json.dump(self.data, f, indent=2)

    def reset(self):
        self.data = []
        self.tags = set()
        self.save()
    
    def add(self, item: MemoryItem):
        tag, fact = item.tag, item.fact
        self.tags.add(tag)
        self.data.append({"tag": tag, "fact": fact})

    def __len__(self):
        return len(self.data)
    
    def retrieve(self, query: str, topk: int = 3) -> List[dict]:
        """Simple keyword-based retrieval."""
        
        query_words = tokenize(query)
        retrieved = []
        
        for memory in reversed(self.data):  # <1>
            tag, fact = memory["tag"], memory["fact"]
            fact = ' '.join(tokenize(fact))
            memory_text = f"{tag} {fact}".lower()

            for word in query_words:
                if word in memory_text: # <2>
                    retrieved.append(memory)
                    break
            
            if len(retrieved) == topk:
                break
        
        return retrieved


client = OpenAI()
mem = MemoryStore()

1. More recent = more relevant.
2. Substring check. e.g. `'commute' in 'commute_experience'` evaluates to `True`.

Next, we define the **generation step** and the **write step**:

In [ ]:
prompt_template = lambda memories, tags: f"""
You are an assistant that processes daily user logs. For each log, extract a concise, 
factual summary of what happened. Each summary should be atomic, standalone, and 
likely useful for future interactions. Assign a relevant `tag` to each summary (`fact`) 
before saving it to memory. 

The following are relevant entries (based on the current input) in the Memory Store:
{memories}

The following are the current tags:
{tags}

**GUIDELINES:**

1. **EXTRACT ATOMIC FACTS:**
    - Break down information into the smallest meaningful, self-contained units.
    - **Good**: "User's favorite programmer is Jon Blow."
    - **Bad**: "User mentioned their favorite programmer is Jon Blow who is a famous game programmer" (This has two facts.)
    - The `fact` must be a concise, direct paraphrase of the fact. Remove conversational fluff.
    - **Good Info:** "User's favorite city is Tokyo"
    - **Bad Info:** "The user stated that if they had to pick a favorite city, they think it would be Tokyo."

2.  **TAG EFFECTIVELY:**
    - **Format:** Prefer generic, descriptive tags in `snake_case`.
    - **Simple:** Prefer simple tags. Choose `commute` is better than `commute_experience`.
    - **Reuse:** Strongly prefer existing tags. Create a new tag only if necessary.
    - An example: For "I really enjoy hiking in the Alps every summer," a good tag is `hobby` or `outdoor_activity`.
    
4.  **EVALUATE & DECIDE:**
    - It is acceptable to save zero logs from an input if nothing is meaningfully new or relevant.
    - Save multiple logs if the user provides multiple distinct pieces of information.
    - Each memory item should make sense on its own. There should be no dependence between separate logs.
"""


def capture_memorable_facts(user_log: str, topk: int=3) -> MemoryResponse:
    """Generate memorable facts from log and write them to memory."""

    # generate memorable facts based on relevant items
    retrieved_memories = mem.retrieve(user_log, topk)
    current_tags = list(mem.tags)
    system_prompt = prompt_template(retrieved_memories, current_tags)

    response = client.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_log},
        ],
        response_format=MemoryResponse,
    )
    
    # save items to memory store
    out = response.choices[0].message.parsed
    for item in out.items:
        mem.add(item)

    mem.save()
    return out

Examples:

In [ ]:
import pandas as pd

logs = [
    "Woke up later than usual because I forgot to set an alarm, rushed through a quick shower, skipped coffee, and still managed to leave for work on time.",
    "Traffic was unusually light, but halfway through I realized I left my ID at home, debated turning back, and decided to just explain at the office front desk.",
    "Took the train, found no seats since it was packed, but struck up a short conversation with a stranger about the book they were reading while we both stood.",
    "Stopped by the bakery, picked up bread for the team, and noticed they had a new seasonal pastry that tempted me but I decided to pass.",
    "Opened my email first thing at the office, skimmed through a pile of routine messages, flagged one urgent client request, and forwarded it to the team lead.",
    "Woah! While crossing the street a man suddenly darted into traffic, cars honked, everyone gasped, and I stood frozen for a moment before hurrying away still shaken.",
    "Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.",
    "Grabbed a pen from my drawer because mine ran out of ink, ended up reorganizing the entire drawer, and discovered an old sticky note with a reminder I had long forgotten."
]

items = []
for log in logs:
    memory_items = capture_memorable_facts(log).items
    for item in memory_items:
        d = item.model_dump()
        d["text"] = log
        items.append(d)

df_resp = pd.DataFrame(items)
mem.reset()

The agent decides whether to reuse a tag or create a new one based on the data:

In [ ]:
#| code-fold: true
import warnings
warnings.simplefilter("ignore")
pd.set_option('display.max_colwidth', None)

print(f"({len(logs)} total logs, {len(df_resp)} facts saved, {len(df_resp.tag.unique())} tags)")
print("tags:")
pprint(list(df_resp.tag.unique()))
df_resp[["tag", "fact", "text", "reason"]]

(8 total logs, 10 facts saved, 8 tags)
tags:
['commute',
 'food_purchase',
 'work_task',
 'unexpected_incident',
 'media_consumption',
 'book_interest',
 'workspace_organization',
 'personal_discovery']


,tag,fact,text,reason
0,commute,"User managed to leave for work on time despite waking up late, skipping coffee, and rushing through a shower.","Woke up later than usual because I forgot to set an alarm, rushed through a quick shower, skipped coffee, and still managed to leave for work on time.",This aligns with existing commute experiences and contributes to understanding daily schedules.
1,commute,User forgot their ID at home but decided to explain at the office front desk instead of turning back.,"Traffic was unusually light, but halfway through I realized I left my ID at home, debated turning back, and decided to just explain at the office front desk.","This fact complements the existing information about the commute, noting the decision-making process and potential impact."
2,food_purchase,User picked up bread for the team at the bakery.,"Stopped by the bakery, picked up bread for the team, and noticed they had a new seasonal pastry that tempted me but I decided to pass.","This log details a specific purchase made by the user, which may be relevant for future preferences or routines."
3,work_task,User opened their email first thing at the office and forwarded an urgent client request to the team lead.,"Opened my email first thing at the office, skimmed through a pile of routine messages, flagged one urgent client request, and forwarded it to the team lead.",The log provides a summary of a specific task conducted by the user as part of their work routine.
4,unexpected_incident,"User witnessed a man dart into traffic, causing cars to honk and bystanders to gasp, which left them shaken.","Woah! While crossing the street a man suddenly darted into traffic, cars honked, everyone gasped, and I stood frozen for a moment before hurrying away still shaken.",This is a distinct and memorable experience regarding an unexpected and alarming incident.
5,media_consumption,User listened to a podcast while walking to the subway.,"Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.","This indicates what the user was doing during their walk, related to media consumption."
6,commute,User was half distracted by construction noise on the street while walking to the subway.,"Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.",This describes part of the user's experience during their commute.
7,book_interest,User made a mental note to check out a book recommended in a podcast.,"Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.",This indicates a new interest or intention to explore a book.
8,workspace_organization,User reorganized their drawer after retrieving a pen.,"Grabbed a pen from my drawer because mine ran out of ink, ended up reorganizing the entire drawer, and discovered an old sticky note with a reminder I had long forgotten.",Recording about workspace organization may provide context for future tasks or preferences.
9,personal_discovery,User discovered an old sticky note with a forgotten reminder in their drawer.,"Grabbed a pen from my drawer because mine ran out of ink, ended up reorganizing the entire drawer, and discovered an old sticky note with a reminder I had long forgotten.",Finding a forgotten reminder may impact user's future decisions or tasks.


llm calls are stateless (all info are in the message history that form its context )

humans include 

- all experiences
- understanding of similar tasks
- reflectsions after encountering past tasks
- what we've learned or have been taught

humans have advanced memory, learn and apply those learning. whereas LLMs do not.

take psych concepts and 

## Farzad

- custom agent with long-term memory
- vector DB, graph DB, with langchain and langgraph
- structure memory into semantic, episodic, and procedural

- 3 chatbots



demo: chat shows memory of past interaction with user (e.g. name) and previous conversations. is able to recall. and collect new information and update its memory.

- llms are stateless. need to constantly repeat context. unable to learn between sessions.
- moreover, memory can improve performance on multi-step tasks.

- databases: graph, sql, vector
    - vector: semantic memory retrieval (topics) 
    - graph: structured relationships, user profiles, inetersts etc.

- summarizing long interactions. fit more context into limited prompt size.
- building agentic loops. tool use, store relevant knowledge over time to guide havior. 

- results in more powerful LLMs. can understand and process longer inputs.

- bottomline: llms are powerful but forgetful memory must be engineered externally to unlock LLM full potential. in practice longer context length results in quadratic growth in memory and time. so infinite context length is not useful in practice.

- memory = tools, databases, algorithm

- LLM context length
```
[
    <instruction (system prompt)>,
    <user info>,    # preference, personality, history
    <chat history summary>,
    <chat history>,
    <tool explanation>,
    <tool calls / results>,
    <few shot examples>,
    <user's question>
]
```

